# nisar-py regression tests

## Running the tests

Use the following command to run the regression test with the default parameters:

```sh
papermill nisar-py_Regression.ipynb output.ipynb
```

Or you can override the default parameters with the `-p` option, e.g:

```sh
papermill nisar-py_Regression.ipynb output.ipynb -p harmony_host_url http://localhost:3000 -p save_md5sums True
```

## Updating the tests

Follow these steps to add or update the md5sum files for new or existing test cases:

1. Edit the code to add any new test cases to the `test_cases` list below.
   Skip this step if you only want to update the md5sums for existing test cases.
2. If you want the notebook to run faster, comment out any existing test cases that you don't want to update.
3. Run the notebook with `-p save_md5sums True`.
   This will cause the notebook to save the md5sums for each test case rather than verify them.
4. Confirm that the expected md5sum files were created or updated.
5. Re-run the notebook *without* `-p save_md5sums True` and confirm the tests pass.
6. Uncomment any existing test cases that you commented out, then commit and push the added/modified md5sum files.

In [ ]:
import hashlib
import json
from pathlib import Path
from tempfile import TemporaryDirectory

import harmony

# Parameters:

harmony_host_url = "https://harmony.uat.earthdata.nasa.gov"

save_md5sums = False

In [ ]:
environments = {
    "http://localhost:3000": harmony.Environment.LOCAL,
    "https://harmony.sit.earthdata.nasa.gov": harmony.Environment.SIT,
    "https://harmony.uat.earthdata.nasa.gov": harmony.Environment.UAT,
    "https://harmony.earthdata.nasa.gov": harmony.Environment.PROD,
}
assert harmony_host_url in environments

harmony_client = harmony.Client(env=environments[harmony_host_url])

harmony_request_params = {
    "collection": harmony.Collection(id="C1273831195-ASF"),
    "labels": ["nisar-py-rtests"],
}

test_cases = [
    {
        # original test case
        "granule_name": "NISAR_L2_PR_GCOV_015_156_A_010_2005_DVDV_A_20230619T000803_20230619T000818_T05000_N_P_J_001",
        "format": "image/png",
    },
    {
        # 20m north polar QP granule in png output (interior Alaska)
        "granule_name": "NISAR_L2_PR_GCOV_008_145_D_054_2005_QPDH_A_20251226T061551_20251226T061609_X05010_N_P_J_001",
        "format": "image/png",
    },
    {
        # 10m utm DH granule in png output (coast of Yemen)
        "granule_name": "NISAR_L2_PR_GCOV_010_136_D_082_4005_DHDH_A_20260118T153232_20260118T153308_X05010_N_F_J_001",
        "format": "image/png",
    },
    {
        # 80m frequencyB DV granule in png output (gulf of mexico)
        "granule_name": "NISAR_L2_PR_GCOV_010_156_D_078_0005_NADV_A_20260120T004801_20260120T004836_X05010_N_F_J_001",
        "format": "image/png",
    },
    {
        # 40m DH granule In geotiff output (new zealand)
        "granule_name": "NISAR_L2_PR_GCOV_004_051_A_155_2005_DHDH_A_20251101T184355_20251101T184430_X05009_N_F_J_001",
        "format": "image/tiff",
    },
]
for test_case in test_cases[3:]: # TODO: revert
    request = harmony.Request(
        **test_case,
        **harmony_request_params,
    )
    job_id = harmony_client.submit(request)
    harmony_client.wait_for_processing(job_id)

    with TemporaryDirectory() as temp_dir:
        output_files = [
            Path(harmony_client.download(url, temp_dir).result())
            for url in harmony_client.result_urls(job_id)
            if not url.endswith(".txt")
        ]
        actual_md5sums = {
            # file extension: md5sum
            file.name.split(".", maxsplit=1)[1]: hashlib.md5(
                file.read_bytes()
            ).hexdigest()
            for file in output_files
        }

    md5sums_path = Path("md5sums") / f"{test_case['granule_name']}.json"
    if save_md5sums:
        print(f"Saving md5sums to {md5sums_path}")
        json.dump(actual_md5sums, md5sums_path.open("w"), indent=4)
    else:
        print(f"Verifying existing md5sums for {test_case['granule_name']}")
        expected_md5sums = json.load(md5sums_path.open())
        assert actual_md5sums == expected_md5sums, (
            f"md5sums for {test_case['granule_name']} do not match expected"
        )

request = harmony.Request(
    # 20m north polar SH
    granule_name="NISAR_L2_PR_GCOV_010_112_A_045_7700_SHNA_A_20260116T231413_20260116T231433_X05010_N_P_J_001",
    format="image/png",
    **harmony_request_params,
)
job_id = harmony_client.submit(request)
try:
    harmony_client.wait_for_processing(job_id)
except harmony.client.ProcessingFailedException as e:
    assert "is single-pol" in str(e), str(e)